In [0]:
# CELL 1: CONFIGURATION, SECURITY & READ SILVER LAYER
# ---------------------------------------------------
from pyspark.sql.functions import col, date_format, months_between, when, round, lit, year, count, sum as spark_sum, avg, last
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType

# 1. SECURITY: Authenticate using Azure Key Vault
storage_account_name = "dlpeopleanalytics2026"
container_name = "medallion-data"

storage_account_key = dbutils.secrets.get(scope="kv-secrets", key="storage-account-key")

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key
)

# 2. PATHS: Define Silver and Gold base paths
silver_base_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/silver"
gold_base_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/gold"

# 3. READ: Load Silver Tables (Reading the full clean history)
df_headcount_silver = spark.read.format("delta").load(f"{silver_base_path}/headcount")
df_roles_silver = spark.read.format("delta").load(f"{silver_base_path}/dim_role")
df_salaries_silver = spark.read.format("delta").load(f"{silver_base_path}/reference_salaries")
df_survey_silver = spark.read.format("delta").load(f"{silver_base_path}/climate_survey")

print("Authentication successful. Full Silver historical data loaded.")

Authentication successful. Full Silver historical data loaded.


In [0]:
# CELL 2: BUILD GOLD EMPLOYEE SNAPSHOT (DETAILED TABLE WITH LOCF)
# ---------------------------------------------------------------
from pyspark.sql.functions import col, date_format, months_between, when, round, lit, year, last
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, DateType

# 1. Base Join: Headcount + Roles + Market Salaries
df_salaries_aligned = df_salaries_silver.drop("seniority", "department", "role_name")

df_base = df_headcount_silver \
    .join(df_roles_silver, ["role_id", "snapshot_date"], "left") \
    .join(df_salaries_aligned, ["role_id", "snapshot_date"], "left")

# 2. Join Surveys
df_survey_subset = df_survey_silver.select(
    "employee_id", 
    "snapshot_date", 
    col("general_satisfaction_score").alias("survey_score")
)

df_base_with_survey = df_base.join(df_survey_subset, ["employee_id", "snapshot_date"], "left")

# 3. Apply LOCF (Last Observation Carried Forward)
window_locf = Window.partitionBy("employee_id").orderBy("snapshot_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_locf = df_base_with_survey.withColumn(
    "satisfaction_score",
    last(col("survey_score"), ignorenulls=True).over(window_locf)
)

# 3.5 Actual response flag (before LOCF covers the data)
df_locf = df_locf.withColumn(
    "is_survey_response",
    when(col("survey_score").isNotNull(), lit(True)).otherwise(lit(False))
)

# 4. Final Formatting, DoubleType Casting and Metrics Calculation
df_gold_snapshot = df_locf \
    .withColumn("market_salary", ((col("max_market_salary") + col("min_market_salary")) / 2)) \
    .select(
        col("snapshot_date").cast(DateType()),
        date_format(col("snapshot_date"), "yyyy-MM").alias("año_mes"),
        year(col("snapshot_date")).alias("año"),
        col("employee_id"),
        col("department"),
        col("role_name"),
        col("seniority"),
        col("gender"),
        round(months_between(col("snapshot_date"), col("hire_date")), 1).cast(DoubleType()).alias("tenure_months"),
        col("current_salary").cast(DoubleType()),
        col("market_salary").cast(DoubleType()),
        round((col("current_salary") / col("market_salary")), 4).cast(DoubleType()).alias("compa_ratio"),
        col("satisfaction_score").cast(DoubleType()),
        col("is_survey_response"),
        col("hire_date").cast(DateType()),
        col("termination_date").cast(DateType())
    )

# 5. Flight Risk Categorization
df_gold_snapshot = df_gold_snapshot.withColumn(
    "flight_risk_category",
    when((col("compa_ratio") < 0.85) & (col("satisfaction_score") <= 3), lit("Alto"))
    .when((col("compa_ratio") < 0.85) | (col("satisfaction_score") <= 3), lit("Medio"))
    .otherwise(lit("Bajo"))
)

print("Gold Employee Monthly Snapshot built. Bug 10604 resolved using DoubleType.")

Gold Employee Monthly Snapshot built. Bug 10604 resolved using DoubleType.


In [0]:
# CELL 3: BUILD GOLD DEPARTMENT KPIS (AGGREGATED TABLE)
# -----------------------------------------------------

# Generate the aggregated metrics
df_gold_kpis = df_gold_snapshot.groupBy("snapshot_date", "año_mes", "año", "department").agg(
    count("employee_id").alias("headcount_total"),
    
    spark_sum(when(date_format(col("hire_date"), "yyyy-MM") == col("año_mes"), 1).otherwise(0)).alias("new_hires"),
    
    spark_sum(when(date_format(col("termination_date"), "yyyy-MM") == col("año_mes"), 1).otherwise(0)).alias("terminations"),
    
    spark_sum("current_salary").cast(DoubleType()).alias("masa_salarial_total"),
    
    round(avg("compa_ratio"), 4).cast(DoubleType()).alias("avg_compa_ratio"),
    round(avg("satisfaction_score"), 2).cast(DoubleType()).alias("avg_satisfaction")
)

# Calculate Turnover Rate
df_gold_kpis = df_gold_kpis.withColumn(
    "turnover_rate_pct", 
    round((col("terminations") / col("headcount_total")) * 100, 2).cast(DoubleType())
)

print("Gold Department KPIs aggregated securely.")

Gold Department KPIs aggregated securely.


In [0]:
# CELL 4: WRITE TO GOLD LAYER (INCREMENTAL MERGE)
# -----------------------------------------------
from delta.tables import DeltaTable

# FIX: allow schema evolution during MERGE, since this run adds a new column
# (is_survey_response) to a Delta table that already exists with the old schema.
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# 1. Define specific Delta table paths
snapshot_path = f"{gold_base_path}/gold_employee_monthly_snapshot"
kpi_path = f"{gold_base_path}/gold_department_monthly_kpi"

# 2. MERGE FOR EMPLOYEE SNAPSHOT
# Check if the table exists (it should, after your historical load)
if DeltaTable.isDeltaTable(spark, snapshot_path):
    delta_snapshot = DeltaTable.forPath(spark, snapshot_path)
    
    # Merge using primary keys and partition key for performance optimization
    delta_snapshot.alias("target").merge(
        df_gold_snapshot.alias("source"),
        "target.employee_id = source.employee_id AND target.snapshot_date = source.snapshot_date AND target.`año` = source.`año`"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    # Fallback mode in case the table was accidentally deleted
    print("Snapshot table not found. Performing initial overwrite creation...")
    df_gold_snapshot.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy("año").save(snapshot_path)

# 3. MERGE FOR DEPARTMENT KPIs
if DeltaTable.isDeltaTable(spark, kpi_path):
    delta_kpi = DeltaTable.forPath(spark, kpi_path)
    
    # Merge using department, date, and partition key
    delta_kpi.alias("target").merge(
        df_gold_kpis.alias("source"),
        "target.department = source.department AND target.snapshot_date = source.snapshot_date AND target.`año` = source.`año`"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    # Fallback mode
    print("KPI table not found. Performing initial overwrite creation...")
    df_gold_kpis.write.format("delta").mode("overwrite").partitionBy("año").save(kpi_path)

print("Gold Layer materialization complete. Incremental MERGE executed successfully.")

Snapshot table not found. Performing initial overwrite creation...
KPI table not found. Performing initial overwrite creation...
Gold Layer materialization complete. Incremental MERGE executed successfully.


In [0]:
# Managerial visualization: compa-ratio vs. satisfaction by department, latest month
latest_month = df_gold_kpis.agg({"snapshot_date": "max"}).collect()[0][0]

display(
    df_gold_kpis
    .filter(col("snapshot_date") == latest_month)
    .select("department", "avg_compa_ratio", "avg_satisfaction", "headcount_total")
    .orderBy("department")
)

department,avg_compa_ratio,avg_satisfaction,headcount_total
Commercial,0.949,3.4,380
Executive,1.0624,3.7,1
Finance,0.9483,3.32,163
Human Resources,0.9528,3.38,91
It & Data,0.9504,3.2,83
Operations,0.9534,3.37,964
Supply Chain,0.9513,3.36,236
